# AI Business Analyst Agent

**Purpose:** Proactive intelligence layer that synthesizes business KPIs, ML predictions, and monitoring signals into executive-ready decision briefs.

**Architecture:**
```
Gold Tables (DuckDB)  →  Deterministic Analytics  →  Structured Signals  →  LangGraph Agent  →  Executive Brief
```

**Design Principle:** The LLM never computes metrics. Python and DuckDB handle all analytics. The LLM reasons over pre-computed, evidence-backed signals and generates narratives.

**Inputs:**
- 5 Gold Parquet files (`customer_360`, `product_performance`, `inventory_health`, `fulfillment_metrics`, `funnel_analytics`)
- `predictions.parquet` (1,000-row inference batch from MLflow)
- `monitoring_summary.json` (Evidently drift flags)
- MLflow model registry metadata

**Outputs:**
- Structured executive briefing (`reports/executive_brief.md`)
- Supporting charts (`reports/charts/`)
- Interactive follow-up capability (Mode 2)


### Configuration

In [19]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain_google_vertexai")
warnings.filterwarnings("ignore", category=FutureWarning)

import json
import os
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import duckdb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

# ── GCP / Gemini ──
PROJECT_ID = "ecommerce-lakehouse-mlops"
LOCATION = "us-central1"
MODEL_NAME = "gemini-2.5-flash-lite"  # swap to "gemini-2.5-flash" for richer narratives

# ── Analysis ──
COMPARISON_TYPE = "month_over_month"
CRITICAL_THRESHOLD = 0.10 
WARNING_THRESHOLD  = 0.05

# ── Paths ──
DATA_DIR    = Path("data")
REPORTS_DIR = Path("reports")
CHARTS_DIR  = REPORTS_DIR / "charts"
EVIDENTLY_DIR = Path("evidently")

REPORTS_DIR.mkdir(exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Gold table files ──
GOLD_TABLES = {
    "customer_360":        DATA_DIR / "customer_360.parquet",
    "product_performance": DATA_DIR / "product_performance.parquet",
    "inventory_health":    DATA_DIR / "inventory_health.parquet",
    "fulfillment_metrics": DATA_DIR / "fulfillment_metrics.parquet",
    "funnel_analytics":    DATA_DIR / "funnel_analytics.parquet",
}
PREDICTIONS_PATH  = DATA_DIR / "predictions.parquet"
MONITORING_PATH   = EVIDENTLY_DIR / "monitoring_summary.json"

# ── Verify all files exist ──
missing = [str(p) for p in list(GOLD_TABLES.values()) + [PREDICTIONS_PATH, MONITORING_PATH] if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

print("=" * 60)
print("  AI BUSINESS ANALYST — CONFIGURATION")
print("=" * 60)
print(f"  Model:          {MODEL_NAME}")
print(f"  Comparison:     {COMPARISON_TYPE}")
print(f"  Critical:       ≥{CRITICAL_THRESHOLD:.0%} change")
print(f"  Warning:        ≥{WARNING_THRESHOLD:.0%} change")
print(f"  Gold tables:    {len(GOLD_TABLES)}")
print(f"  Reports:        {REPORTS_DIR}")
print("=" * 60)

  AI BUSINESS ANALYST — CONFIGURATION
  Model:          gemini-2.5-flash-lite
  Comparison:     month_over_month
  Critical:       ≥10% change
  Gold tables:    5
  Reports:        reports


### Domain Models

Typed interfaces between the analytics layer and the agent. Every finding carries its own evidence and the SQL query that produced it, so any claim in the executive brief can be traced back to a deterministic computation.


In [20]:
@dataclass
class Signal:
    """A single KPI measurement with period-over-period comparison."""
    metric: str
    domain: str                  # e.g. "revenue", "customers", "fulfillment"
    current_value: float
    prior_value: float
    delta_pct: float             # percentage change (signed)
    severity: str                # "critical", "warning", "normal"
    direction: str               # "up", "down", "flat"
    context: str                 # human-readable one-liner
    source_query: str = ""       # the DuckDB SQL that produced this


@dataclass
class Finding:
    """An anomaly or noteworthy pattern that deserves executive attention."""
    title: str
    description: str
    confidence: float            # 0.0–1.0, computed from corroborating evidence
    severity: str                # "critical", "warning", "info"
    evidence: list = field(default_factory=list)    # list of evidence strings
    source_query: str = ""       # reproducible SQL
    recommended_action: str = ""


@dataclass
class ExecutiveBrief:
    """The final structured output the LLM generates."""
    executive_summary: str = ""
    business_health: str = ""
    key_findings: str = ""
    model_health: str = ""
    recommended_actions: str = ""
    generated_at: str = ""


print("Domain models defined: Signal, Finding, ExecutiveBrief")

Domain models defined: Signal, Finding, ExecutiveBrief


### Data Platform Integration

Load all Gold tables into DuckDB as a lightweight local SQL engine. In production, this would be a Databricks SQL warehouse or a Spark session. The agent's architecture is identical, only the connection string changes.


In [21]:
# ── Initialize DuckDB in-memory database ──
con = duckdb.connect(":memory:")

# ── Register Gold Parquet files as queryable tables ──
for table_name, parquet_path in GOLD_TABLES.items():
    con.execute(f"CREATE TABLE {table_name} AS SELECT * FROM read_parquet('{parquet_path}')")

# ── Register predictions ──
con.execute(f"CREATE TABLE predictions AS SELECT * FROM read_parquet('{PREDICTIONS_PATH}')")

# ── Load monitoring summary ──
with open(MONITORING_PATH) as f:
    monitoring_summary = json.load(f)

# ── Verify table registration ──
tables = con.execute("SHOW TABLES").fetchall()
print(f"DuckDB tables registered: {len(tables)}")
for t in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {t[0]}").fetchone()[0]
    print(f"  {t[0]:25s} → {count:>10,} rows")

print(f"\nMonitoring summary loaded: recommendation = {monitoring_summary['recommendation']}")

DuckDB tables registered: 6
  customer_360              →    100,000 rows
  fulfillment_metrics       →        903 rows
  funnel_analytics          →    124,082 rows
  inventory_health          →     29,036 rows
  predictions               →      1,000 rows
  product_performance       →     29,120 rows

Monitoring summary loaded: recommendation = INVESTIGATE


### Deterministic Analytics

Pure Python and DuckDB. No LLM. This section computes all business KPIs with period-over-period comparisons. Every metric is computed from a SQL query that can be independently verified.


In [22]:
def compute_signal(metric: str, domain: str, current: float, prior: float, query: str = "") -> Signal:
    """Compute a Signal from current vs prior values with direction-aware severity."""
    if prior == 0:
        delta_pct = 0.0
    else:
        delta_pct = (current - prior) / abs(prior)

    abs_delta = abs(delta_pct)

    if delta_pct > 0.005:
        direction = "up"
    elif delta_pct < -0.005:
        direction = "down"
    else:
        direction = "flat"

    # ── Direction-aware severity ──
    # Large negative changes are critical (something is wrong).
    # Large positive changes are notable (good news, but understand why).
    # This prevents revenue going UP 22% from being flagged as a crisis.
    if abs_delta >= CRITICAL_THRESHOLD:
        severity = "critical" if direction == "down" else "notable"
    elif abs_delta >= WARNING_THRESHOLD:
        severity = "warning"
    else:
        severity = "normal"

    context = f"{metric}: {current:,.2f} (was {prior:,.2f}), {delta_pct:+.1%} {direction}"
    return Signal(metric=metric, domain=domain, current_value=current, prior_value=prior,
                  delta_pct=delta_pct, severity=severity, direction=direction,
                  context=context, source_query=query)


def run_analytics(con) -> list:
    """Run all KPI computations and return a list of Signals."""
    signals = []

    # ── Determine analysis periods ──
    # Use fulfillment_metrics which has a natural month grain (order_month)
    period_query = """
    SELECT DISTINCT order_month
    FROM fulfillment_metrics
    ORDER BY order_month DESC
    LIMIT 2
    """
    periods = con.execute(period_query).fetchall()
    if len(periods) < 2:
        print("WARNING: fewer than 2 periods found, cannot compute deltas")
        return signals

    current_period = periods[0][0]
    prior_period = periods[1][0]
    print(f"  Analysis window: {prior_period} → {current_period} ({COMPARISON_TYPE})")

    # ── 1. Revenue ──
    rev_query = """
    SELECT
        SUM(CASE WHEN order_month = ? THEN total_revenue ELSE 0 END) AS current_rev,
        SUM(CASE WHEN order_month = ? THEN total_revenue ELSE 0 END) AS prior_rev
    FROM fulfillment_metrics
    """
    row = con.execute(rev_query, [current_period, prior_period]).fetchone()
    signals.append(compute_signal("Total Revenue", "revenue", row[0] or 0, row[1] or 0, rev_query))

    # ── 2. Orders Fulfilled ──
    orders_query = """
    SELECT
        SUM(CASE WHEN order_month = ? THEN orders_fulfilled ELSE 0 END) AS cur,
        SUM(CASE WHEN order_month = ? THEN orders_fulfilled ELSE 0 END) AS prior
    FROM fulfillment_metrics
    """
    row = con.execute(orders_query, [current_period, prior_period]).fetchone()
    signals.append(compute_signal("Orders Fulfilled", "revenue", row[0] or 0, row[1] or 0, orders_query))

    # ── 3. Active Customers (customers with orders in period) ──
    active_query = """
    SELECT
        (SELECT COUNT(DISTINCT user_id) FROM funnel_analytics
         WHERE event_month = ?) AS cur,
        (SELECT COUNT(DISTINCT user_id) FROM funnel_analytics
         WHERE event_month = ?) AS prior
    """
    row = con.execute(active_query, [current_period, prior_period]).fetchone()
    signals.append(compute_signal("Active Users", "customers", row[0] or 0, row[1] or 0, active_query))

    # ── 4. Avg Conversion Rate ──
    conv_query = """
    SELECT
        AVG(CASE WHEN event_month = ? THEN session_to_purchase_rate END) AS cur,
        AVG(CASE WHEN event_month = ? THEN session_to_purchase_rate END) AS prior
    FROM funnel_analytics
    """
    row = con.execute(conv_query, [current_period, prior_period]).fetchone()
    signals.append(compute_signal("Avg Conversion Rate", "conversion", row[0] or 0, row[1] or 0, conv_query))

    # ── 5. Fulfillment: Avg Delivery Days ──
    del_query = """
    SELECT
        AVG(CASE WHEN order_month = ? THEN avg_delivery_days END) AS cur,
        AVG(CASE WHEN order_month = ? THEN avg_delivery_days END) AS prior
    FROM fulfillment_metrics
    """
    row = con.execute(del_query, [current_period, prior_period]).fetchone()
    signals.append(compute_signal("Avg Delivery Days", "fulfillment", row[0] or 0, row[1] or 0, del_query))

    # ── 6. On-Time Rate ──
    ot_query = """
    SELECT
        AVG(CASE WHEN order_month = ? THEN on_time_rate END) AS cur,
        AVG(CASE WHEN order_month = ? THEN on_time_rate END) AS prior
    FROM fulfillment_metrics
    """
    row = con.execute(ot_query, [current_period, prior_period]).fetchone()
    signals.append(compute_signal("On-Time Delivery Rate", "fulfillment", row[0] or 0, row[1] or 0, ot_query))

    # ── 7. Inventory: Dead Stock Rate ──
    dead_query = """
    SELECT
        AVG(CASE WHEN is_dead_stock = true THEN 1.0 ELSE 0.0 END) AS dead_rate,
        COUNT(*) AS total_products
    FROM inventory_health
    """
    row = con.execute(dead_query).fetchone()
    dead_signal = Signal(
        metric="Dead Stock Rate", domain="inventory",
        current_value=row[0] or 0, prior_value=0, delta_pct=0,
        severity="info" if (row[0] or 0) < 0.10 else "warning",
        direction="flat", context=f"Dead stock rate: {(row[0] or 0):.1%} across {row[1]:,} products",
        source_query=dead_query
    )
    signals.append(dead_signal)

    # ── 8. Churn Rate (from predictions) ──
    churn_query = """
    SELECT
        AVG(churn_probability) AS avg_prob,
        COUNT(CASE WHEN churn_probability > 0.7 THEN 1 END) AS high_risk,
        COUNT(*) AS total
    FROM predictions
    """
    row = con.execute(churn_query).fetchone()
    high_risk_pct = (row[1] or 0) / max(row[2] or 1, 1)
    churn_signal = Signal(
        metric="Churn Risk (Inference Batch)", domain="churn",
        current_value=row[1] or 0, prior_value=row[2] or 0,
        delta_pct=high_risk_pct,
        severity="warning" if high_risk_pct > 0.5 else "normal",
        direction="up" if high_risk_pct > 0.5 else "flat",
        context=f"Churn risk: {row[1] or 0} of {row[2] or 0} customers high-risk (prob > 0.7), avg prob {(row[0] or 0):.3f}",
        source_query=churn_query
    )
    signals.append(churn_signal)

    return signals, current_period, prior_period


# ── Run analytics ──
print("Running deterministic analytics...")
signals, current_period, prior_period = run_analytics(con)
print(f"\nComputed {len(signals)} signals:\n")
for s in signals:
    icons = {"critical": "🔴", "notable": "📈", "warning": "🟡", "normal": "🟢", "info": "ℹ️"}
    icon = icons.get(s.severity, "⚪")
    print(f"  {icon} {s.context}")


Running deterministic analytics...
  Analysis window: 2026-06-01 00:00:00 → 2026-07-01 00:00:00 (month_over_month)

Computed 8 signals:

  📈 Total Revenue: 629,420.47 (was 513,446.07), +22.6% up
  📈 Orders Fulfilled: 10,366.00 (was 8,313.00), +24.7% up
  🟢 Active Users: 5,800.00 (was 5,606.00), +3.5% up
  🟢 Avg Conversion Rate: 1.00 (was 0.97), +2.6% up
  🟢 Avg Delivery Days: 2.96 (was 3.06), -3.3% down
  🟢 On-Time Delivery Rate: 0.99 (was 0.99), +0.3% flat
  ℹ️ Dead stock rate: 7.5% across 29,036 products
  🟢 Churn risk: 394 of 1000 customers high-risk (prob > 0.7), avg prob 0.606


### Signal Generation & Finding Detection

Evaluate which signals deserve executive attention. Build structured Findings with confidence scores and evidence lists. The LLM will reason over these findings, it will never compute them.


In [23]:
def generate_findings(signals: list, con, current_period, prior_period) -> list:
    """Analyze signals, detect anomalies, and produce evidence-backed Findings."""
    findings = []

    # ── Collect actionable signals (critical or notable) and warnings ──
    actionable_signals = [s for s in signals if s.severity in ("critical", "notable")]
    warning_signals = [s for s in signals if s.severity == "warning"]

    # ── For each actionable signal, investigate deeper ──
    for sig in actionable_signals:
        evidence = [sig.context]
        drill_query = ""

        if sig.domain == "revenue":
            drill_query = f"""
            SELECT distribution_center_name, total_revenue,
                   orders_fulfilled
            FROM fulfillment_metrics
            WHERE order_month = '{current_period}'
            ORDER BY total_revenue DESC
            LIMIT 5
            """
            rows = con.execute(drill_query).fetchall()
            for r in rows:
                evidence.append(f"  DC '{r[0]}': ${r[1]:,.0f} revenue, {r[2]:,} orders")

        elif sig.domain == "customers":
            drill_query = """
            SELECT traffic_source, COUNT(*) AS users,
                   AVG(order_count) AS avg_orders
            FROM customer_360
            GROUP BY traffic_source
            ORDER BY users DESC
            """
            rows = con.execute(drill_query).fetchall()
            for r in rows:
                evidence.append(f"  Source '{r[0]}': {r[1]:,} users, avg {r[2]:.1f} orders")

        elif sig.domain == "conversion":
            drill_query = f"""
            SELECT event_month, AVG(session_to_purchase_rate) AS avg_conv,
                   COUNT(DISTINCT user_id) AS users
            FROM funnel_analytics
            GROUP BY event_month
            ORDER BY event_month DESC
            LIMIT 6
            """
            rows = con.execute(drill_query).fetchall()
            for r in rows:
                evidence.append(f"  {r[0]}: conv={r[1]:.4f}, users={r[2]:,}")

        elif sig.domain == "fulfillment":
            drill_query = f"""
            SELECT distribution_center_name,
                   AVG(avg_delivery_days) AS avg_del,
                   AVG(on_time_rate) AS avg_ot
            FROM fulfillment_metrics
            WHERE order_month = '{current_period}'
            GROUP BY distribution_center_name
            ORDER BY avg_del DESC
            LIMIT 5
            """
            rows = con.execute(drill_query).fetchall()
            for r in rows:
                evidence.append(f"  DC '{r[0]}': {r[1]:.1f} days avg, {r[2]:.1%} on-time")

        # ── Compute confidence from corroborating signals ──
        domain_signals = [s for s in signals if s.domain == sig.domain]
        corroborating = [s for s in domain_signals if s.severity in ("critical", "notable", "warning")]
        confidence = min(len(corroborating) / max(len(domain_signals), 1), 1.0)
        if len(evidence) > 2:
            confidence = min(confidence + 0.2, 1.0)

        findings.append(Finding(
            title=f"{sig.metric} — {sig.direction.upper()} {abs(sig.delta_pct):.1%}",
            description=f"{sig.metric} moved {sig.delta_pct:+.1%} from {prior_period} to {current_period}.",
            confidence=round(confidence, 2),
            severity=sig.severity,
            evidence=evidence,
            source_query=drill_query or sig.source_query,
            recommended_action=f"Investigate {sig.metric.lower()} drivers in the {sig.domain} domain."
        ))

    # ── Add warning-level findings ──
    for sig in warning_signals:
        findings.append(Finding(
            title=f"{sig.metric} — {sig.direction.upper()} {abs(sig.delta_pct):.1%}",
            description=sig.context,
            confidence=0.6,
            severity="warning",
            evidence=[sig.context],
            source_query=sig.source_query,
            recommended_action=f"Monitor {sig.metric.lower()} trend over the next period."
        ))

    # ── Add monitoring finding (always present) ──
    mon = monitoring_summary
    mon_severity = "warning" if mon.get("prediction_drift_detected") else "normal"

    # Compute monitoring confidence from how many flags are raised
    mon_flags = [
        mon.get("data_drift_detected", False),
        mon.get("prediction_drift_detected", False),
        mon.get("model_degraded", False),
    ]
    flags_raised = sum(1 for f in mon_flags if f)
    mon_confidence = round(0.7 + (flags_raised * 0.1), 2)  # 0.7 base + 0.1 per flag

    findings.append(Finding(
        title="ML Model Monitoring Status",
        description=(
            f"Data drift: {mon.get('features_drifted', 0)}/{mon.get('features_tested', 0)} features. "
            f"Prediction drift: {'YES' if mon.get('prediction_drift_detected') else 'No'}. "
            f"Model degradation: {'YES' if mon.get('model_degraded') else 'No'}. "
            f"Recommendation: {mon.get('recommendation', 'N/A')}."
        ),
        confidence=mon_confidence,
        severity=mon_severity,
        evidence=[
            f"Feature drift: {mon.get('features_drifted')}/{mon.get('features_tested')} features drifted",
            f"Prediction drift: {mon.get('prediction_drift_method')} = {mon.get('prediction_drift_score')} (threshold: {mon.get('prediction_drift_threshold')})",
            f"AUC: {mon.get('auc_ref')} → {mon.get('auc_cur')} (drop: {mon.get('auc_drop_pct')}%)",
            f"Recommendation: {mon.get('recommendation')}",
        ],
        source_query="SELECT * FROM monitoring_summary.json",
        recommended_action="Investigate prediction drift source before retraining."
    ))

    return findings


# ── Generate findings ──
findings = generate_findings(signals, con, current_period, prior_period)

# ── Threshold explanation ──
# Findings = promoted signals (critical/notable/warning) + 1 monitoring finding (always injected)
signal_based_findings = len([s for s in signals if s.severity in ("critical", "notable", "warning")])
not_promoted = len(signals) - signal_based_findings
print(f"Generated {len(findings)} findings from {len(signals)} signals:")
print(f"  {signal_based_findings} signals promoted + 1 monitoring finding = {len(findings)} total")
print(f"  ({not_promoted} signals below {WARNING_THRESHOLD:.0%} threshold → not promoted)\n")

for i, f in enumerate(findings, 1):
    icons = {"critical": "🔴", "notable": "📈", "warning": "🟡", "normal": "🟢", "info": "ℹ️"}
    icon = icons.get(f.severity, "⚪")
    print(f"  {icon} [{f.confidence:.0%}] {f.title}")
    for e in f.evidence[:3]:
        print(f"       {e}")
    if len(f.evidence) > 3:
        print(f"       ... and {len(f.evidence) - 3} more")
    print()


Generated 3 findings from 8 signals:
  2 signals promoted + 1 monitoring finding = 3 total
  (6 signals below 5% threshold → not promoted)

  📈 [100%] Total Revenue — UP 22.6%
       Total Revenue: 629,420.47 (was 513,446.07), +22.6% up
         DC 'Houston TX': $92,362 revenue, 1,282 orders
         DC 'Memphis TN': $81,733 revenue, 1,349 orders
       ... and 3 more

  📈 [100%] Orders Fulfilled — UP 24.7%
       Orders Fulfilled: 10,366.00 (was 8,313.00), +24.7% up
         DC 'Houston TX': $92,362 revenue, 1,282 orders
         DC 'Memphis TN': $81,733 revenue, 1,349 orders
       ... and 3 more

  🟡 [80%] ML Model Monitoring Status
       Feature drift: 0/21 features drifted
       Prediction drift: Wasserstein distance (normed) = 5.4203 (threshold: 0.1)
       AUC: 0.6673 → 0.6661 (drop: 0.18%)
       ... and 1 more



### Agent Tool Definitions

Five tools that the LangGraph agent can invoke. None of these tools perform analytics. The tools provide access to pre-computed results and enable follow-up investigation.


In [24]:
from langchain_core.tools import tool


@tool
def kpi_summary() -> str:
    """Return the pre-computed business KPI signals and findings.
    Call this first to understand the current business state."""
    lines = ["=== KPI SIGNALS ===\n"]
    for s in signals:
        status = "CRITICAL" if s.severity == "critical" else "WARNING" if s.severity == "warning" else "OK"
        lines.append(f"[{status}] {s.context}")
    lines.append("\n=== KEY FINDINGS ===\n")
    for f in findings:
        lines.append(f"[{f.severity.upper()} | confidence={f.confidence:.0%}] {f.title}")
        lines.append(f"  {f.description}")
        if f.recommended_action:
            lines.append(f"  → Action: {f.recommended_action}")
    return "\n".join(lines)


@tool
def sql_query(query: str) -> str:
    """Execute a SQL query against the Gold tables in DuckDB.
    Available tables: customer_360, product_performance, inventory_health,
    fulfillment_metrics, funnel_analytics, predictions.
    Return the results as a formatted table.
    Use this for follow-up investigations to drill into specific metrics."""
    try:
        result = con.execute(query).fetchdf()
        if result.empty:
            return "Query returned no results."
        return result.to_string(index=False, max_rows=20, float_format=lambda x: f"{x:,.2f}")
    except Exception as e:
        return f"SQL Error: {e}"


@tool
def churn_analysis() -> str:
    """Return churn prediction analysis from the latest inference batch.
    Includes probability distribution, high-risk segments, and key statistics."""
    stats = con.execute("""
        SELECT
            COUNT(*) AS total_customers,
            AVG(churn_probability) AS avg_probability,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY churn_probability) AS median_probability,
            PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY churn_probability) AS p90_probability,
            COUNT(CASE WHEN churn_probability > 0.7 THEN 1 END) AS high_risk_count,
            COUNT(CASE WHEN churn_probability > 0.9 THEN 1 END) AS very_high_risk_count
        FROM predictions
    """).fetchone()

    return (
        f"Churn Analysis (inference batch of {stats[0]:,} customers):\n"
        f"  Avg probability:     {stats[1]:.4f}\n"
        f"  Median probability:  {stats[2]:.4f}\n"
        f"  P90 probability:     {stats[3]:.4f}\n"
        f"  High risk (>0.7):    {stats[4]:,} ({stats[4]/stats[0]:.1%})\n"
        f"  Very high risk (>0.9): {stats[5]:,} ({stats[5]/stats[0]:.1%})"
    )


@tool
def monitoring_status() -> str:
    """Return the Evidently AI monitoring summary.
    Includes data drift, prediction drift, model quality, and recommendation."""
    m = monitoring_summary
    return (
        f"Model Monitoring Status (as of {m.get('timestamp', 'unknown')}):\n"
        f"  Data drift:        {'DETECTED' if m.get('data_drift_detected') else 'None'} "
        f"({m.get('features_drifted', 0)}/{m.get('features_tested', 0)} features)\n"
        f"  Prediction drift:  {'DETECTED' if m.get('prediction_drift_detected') else 'None'} "
        f"({m.get('prediction_drift_method', 'N/A')} = {m.get('prediction_drift_score', 'N/A')}, "
        f"threshold = {m.get('prediction_drift_threshold', 'N/A')})\n"
        f"  Model quality:     AUC {m.get('auc_ref', 'N/A')} → {m.get('auc_cur', 'N/A')} "
        f"(drop: {m.get('auc_drop_pct', 'N/A')}%)\n"
        f"  Recommendation:    {m.get('recommendation', 'N/A')}"
    )


@tool
def chart_generator(chart_type: str, title: str, data_query: str) -> str:
    """Generate a matplotlib chart from a SQL query and save it.
    Args:
        chart_type: 'bar', 'line', or 'pie'
        title: Chart title
        data_query: SQL query that returns 2 columns (label, value)
    Returns: Path to the saved chart image."""
    try:
        df = con.execute(data_query).fetchdf()
        if df.empty:
            return "No data to chart."

        fig, ax = plt.subplots(figsize=(10, 6))
        cols = df.columns.tolist()

        if chart_type == "bar":
            ax.bar(df[cols[0]].astype(str), df[cols[1]], color="#4A90D9")
        elif chart_type == "line":
            ax.plot(df[cols[0]].astype(str), df[cols[1]], marker="o", color="#4A90D9")
        elif chart_type == "pie":
            ax.pie(df[cols[1]], labels=df[cols[0]].astype(str), autopct="%1.1f%%")
        else:
            return f"Unknown chart type: {chart_type}"

        ax.set_title(title, fontsize=14, fontweight="bold")
        if chart_type != "pie":
            ax.tick_params(axis="x", rotation=45)
            ax.set_ylabel(cols[1])
        plt.tight_layout()

        safe_title = title.lower().replace(" ", "_")[:40]
        chart_path = CHARTS_DIR / f"{safe_title}.png"
        fig.savefig(chart_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        return f"Chart saved: {chart_path}"
    except Exception as e:
        return f"Chart error: {e}"


agent_tools = [kpi_summary, sql_query, churn_analysis, monitoring_status, chart_generator]
print(f"Defined {len(agent_tools)} agent tools: {[t.name for t in agent_tools]}")


Defined 5 agent tools: ['kpi_summary', 'sql_query', 'churn_analysis', 'monitoring_status', 'chart_generator']


### LangGraph Agent

Single agent with conditional branching. Three nodes, one conditional edge:
- **load_signals** — Inject pre-computed signals and findings into the agent state.
- **investigate** — If critical findings exist, drill deeper using tools.
- **generate_brief** — LLM synthesizes all findings into a structured executive briefing.

The graph routes conditionally: critical findings → investigate → brief; no critical findings → brief directly.


In [28]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain")
warnings.filterwarnings("ignore", message=".*was deprecated.*")

from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# ── Initialize LLM ──
llm = ChatVertexAI(
    model_name=MODEL_NAME,
    project=PROJECT_ID,
    location=LOCATION,
    temperature=0.3,
    max_output_tokens=4096,
)

# Bind tools so the LLM can call them during investigation
llm_with_tools = llm.bind_tools(agent_tools)


# ── Agent State ──
class AgentState(TypedDict):
    signals_text: str
    findings_text: str
    has_critical: bool
    investigation_results: str
    briefing: str
    messages: list


# ── Node: Load Signals ──
def load_signals_node(state: AgentState) -> AgentState:
    """Inject pre-computed signals and findings into the agent state."""
    signals_text = "\n".join([s.context for s in signals])

    findings_text_parts = []
    for f in findings:
        findings_text_parts.append(
            f"[{f.severity.upper()} | confidence={f.confidence:.0%}] {f.title}\n"
            f"  {f.description}\n"
            f"  Evidence: {'; '.join(f.evidence[:3])}\n"
            f"  Action: {f.recommended_action}"
        )
    findings_text = "\n\n".join(findings_text_parts)

    has_critical = any(f.severity in ("critical", "notable") for f in findings)

    return {
        **state,
        "signals_text": signals_text,
        "findings_text": findings_text,
        "has_critical": has_critical,
        "investigation_results": "",
    }


# ── Node: Investigate ──
def investigate_node(state: AgentState) -> AgentState:
    """For critical/notable findings, ask the LLM to use tools for deeper investigation."""
    actionable = [f for f in findings if f.severity in ("critical", "notable")]
    investigation_prompt = (
        "You are an AI Business Analyst. The following findings need investigation:\n\n"
        + "\n".join([f"- {f.title}: {f.description}" for f in actionable])
        + "\n\nUse the sql_query and churn_analysis tools to investigate the root causes. "
        "Be specific about which segments, categories, or distribution centers are driving these changes. "
        "Provide your analysis in a structured format."
    )

    messages = [
        SystemMessage(content="You are an expert business analyst. Use the provided tools to investigate."),
        HumanMessage(content=investigation_prompt),
    ]

    response = llm_with_tools.invoke(messages)

    # If the LLM made tool calls, execute them
    investigation_text = ""
    if hasattr(response, "tool_calls") and response.tool_calls:
        for tc in response.tool_calls:
            tool_fn = {t.name: t for t in agent_tools}.get(tc["name"])
            if tool_fn:
                try:
                    result = tool_fn.invoke(tc["args"])
                    investigation_text += f"\n[Tool: {tc['name']}]\n{result}\n"
                except Exception as e:
                    investigation_text += f"\n[Tool: {tc['name']}] Error: {e}\n"
    else:
        investigation_text = response.content if hasattr(response, "content") and response.content else ""

    return {**state, "investigation_results": investigation_text}


# ── Node: Generate Brief ──
def generate_brief_node(state: AgentState) -> AgentState:
    """LLM synthesizes all findings into a structured executive briefing."""
    brief_prompt = f"""You are an AI Business Analyst generating an executive briefing.

BUSINESS SIGNALS:
{state['signals_text']}

KEY FINDINGS:
{state['findings_text']}

{f"INVESTIGATION RESULTS:{chr(10)}{state['investigation_results']}" if state['investigation_results'] else ""}

Generate a structured executive briefing with EXACTLY these 5 sections.
Use Markdown formatting. Be specific — cite numbers from the signals and findings.

STRICT RULES:
- ONLY reference metrics, values, and evidence explicitly provided above.
- Do NOT introduce external explanations, marketing hypotheses, or ungrounded recommendations.
- Every claim must trace to a specific signal or finding listed above.
- If you cannot explain a change from the evidence, say "requires further investigation" rather than guessing.

## Executive Summary
Two to three sentences summarizing the overall business state.

## Business Health
A scoreboard of key metrics with their status (green/yellow/red) and delta values.
Format as a table.

## Key Findings & Hypotheses
The most important findings, each with:
- What happened (cite the specific metric and delta)
- Why it might have happened (hypothesis based ONLY on the evidence above)
- Confidence level

## Model & Monitoring Status
Current state of the ML model and monitoring system.

## Recommended Actions & Open Questions
Three to five specific, actionable recommendations tied to the findings above.
Plus any open questions that cannot be answered from the current data alone.
"""

    messages = [
        SystemMessage(content=(
            "You are an expert business analyst at a major e-commerce company. "
            "Your job is to synthesize pre-computed business signals and findings "
            "into a clear, actionable executive briefing. "
            "Never compute metrics yourself — only reference what is provided. "
            "Never introduce external knowledge or hypotheses not supported by the evidence. "
            "Write in a professional but direct tone. No filler."
        )),
        HumanMessage(content=brief_prompt),
    ]

    response = llm.invoke(messages)
    briefing = response.content if hasattr(response, "content") and response.content else ""

    return {**state, "briefing": briefing}


# ── Routing function ──
def route_by_severity(state: AgentState) -> str:
    """Route to investigation if critical/notable findings exist, else straight to briefing."""
    return "investigate" if state["has_critical"] else "generate_brief"


# ── Build the graph ──
workflow = StateGraph(AgentState)

workflow.add_node("load_signals", load_signals_node)
workflow.add_node("investigate", investigate_node)
workflow.add_node("generate_brief", generate_brief_node)

workflow.add_edge(START, "load_signals")
workflow.add_conditional_edges("load_signals", route_by_severity, {
    "investigate": "investigate",
    "generate_brief": "generate_brief",
})
workflow.add_edge("investigate", "generate_brief")
workflow.add_edge("generate_brief", END)

agent = workflow.compile()
print("LangGraph agent compiled successfully.")
print(f"  Nodes: load_signals → route → [investigate] → generate_brief → END")
has_actionable = any(f.severity in ('critical', 'notable') for f in findings)
print(f"  Routing: {'INVESTIGATE path (actionable findings detected)' if has_actionable else 'DIRECT path (no actionable findings)'}")


LangGraph agent compiled successfully.
  Nodes: load_signals → route → [investigate] → generate_brief → END
  Routing: INVESTIGATE path (actionable findings detected)


/var/folders/yd/f40h2d312dggyb2s4268_n400000gn/T/ipykernel_55525/3193265517.py:11: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(


### Mode 1: Executive Briefing

The agent runs proactively. It scans the business state, identifies what changed, explains why, and recommends actions. The output is a structured executive briefing saved to `reports/executive_brief.md`.


In [26]:
# ── Run the agent ──
print("Generating executive briefing...\n")

initial_state = {
    "signals_text": "",
    "findings_text": "",
    "has_critical": False,
    "investigation_results": "",
    "briefing": "",
    "messages": [],
}

result = agent.invoke(initial_state)

# ── Display the briefing ──
briefing_text = result["briefing"]
print(briefing_text)

# ── Save to markdown ──
header = f"""# Executive Briefing — E-Commerce Intelligence Platform

**Generated:** {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}
**Analysis Period:** {prior_period} → {current_period} ({COMPARISON_TYPE})
**Model:** {MODEL_NAME}
**Findings:** {len(findings)} ({sum(1 for f in findings if f.severity == 'critical')} critical, {sum(1 for f in findings if f.severity == 'warning')} warning)

---

"""

brief_path = REPORTS_DIR / "executive_brief.md"
brief_path.write_text(header + briefing_text, encoding="utf-8")
print(f"\n{'=' * 60}")
print(f"  Briefing saved to: {brief_path}")
print(f"  Route taken: {'investigate → brief' if result.get('investigation_results') else 'direct → brief'}")
print(f"{'=' * 60}")


Generating executive briefing...

## Executive Summary
Total Revenue increased by 22.6% and Orders Fulfilled increased by 24.7%, indicating strong growth in sales volume. Active Users saw a modest increase of 3.5%, with a slight improvement in Avg Conversion Rate. However, a warning has been issued regarding ML model monitoring due to prediction drift, requiring immediate investigation.

## Business Health

| Metric              | Current Value | Previous Value | Change   | Status |
| :------------------ | :------------ | :------------- | :------- | :----- |
| Total Revenue       | 629,420.47    | 513,446.07     | +22.6%   | Green  |
| Orders Fulfilled    | 10,366.00     | 8,313.00       | +24.7%   | Green  |
| Active Users        | 5,800.00      | 5,606.00       | +3.5%    | Green  |
| Avg Conversion Rate | 1.00          | 0.97           | +2.6%    | Green  |
| Avg Delivery Days   | 2.96          | 3.06           | -3.3%    | Green  |
| On-Time Delivery Rate| 0.99          | 0.99     

### Mode 2: Interactive Follow-Up

The briefing establishes context. Follow-up questions are grounded in what the agent already found. This section demonstrates the agent's ability to answer contextual questions using SQL and chart tools.

In the Streamlit dashboard, this becomes an interactive chat bar. Here, we demonstrate the capability with three example questions.


In [ ]:
# ── Table schemas for grounding (prevents schema hallucination) ──
# Includes value-domain hints for columns where the LLM might guess wrong
TABLE_SCHEMAS = """
AVAILABLE TABLES AND THEIR EXACT COLUMN NAMES:

customer_360: user_id, first_name, last_name, email, age, gender (values: M, F),
  country, state, city, traffic_source (values: Search, Organic, Facebook, Email, Display),
  account_created_at, order_count, lifetime_spend, avg_order_value,
  avg_item_sale_price, total_items_purchased, distinct_products_purchased, return_rate,
  favorite_category, first_order_at, last_order_at, days_since_last_order,
  first_to_second_order_days, total_sessions, session_count_30d, days_since_last_session,
  avg_session_depth, browse_to_buy_ratio, cart_abandonment_rate

product_performance: product_id, name, brand, category, department (values: Men, Women),
  sku, cost, retail_price, base_margin, distribution_center_id, units_sold,
  total_revenue, avg_selling_price, unique_buyers, units_returned, return_rate,
  total_units_received, units_in_stock, sell_through_rate, avg_days_to_sell,
  total_margin, margin_pct, category_revenue_rank, first_sold_at, last_sold_at

inventory_health: product_id, name, brand, category, distribution_center_id,
  distribution_center_name, total_units_received, units_sold, units_in_stock,
  avg_unit_cost, stock_value, turnover_rate, avg_days_to_sell, days_of_supply,
  daily_sell_rate, last_sale_at, days_since_last_sale,
  is_dead_stock (boolean: true/false), reorder_signal (boolean: true/false),
  oldest_unsold_received_at

fulfillment_metrics: distribution_center_id, distribution_center_name, order_month,
  orders_fulfilled, items_fulfilled, avg_delivery_days, median_delivery_days,
  avg_ship_days, on_time_deliveries, total_deliveries, on_time_rate, returned_items,
  return_rate, total_revenue

funnel_analytics: user_id, event_month, sessions, total_events, avg_session_depth,
  home_events, department_events, product_view_events, cart_events, purchase_events,
  cancel_events, session_to_purchase_rate, cart_to_purchase_rate, browse_depth,
  sessions_mom_change, first_event_at, last_event_at

predictions: user_id, churn_probability (float 0-1),
  predicted_label (integer: 1=churned, 0=not churned),
  prediction_timestamp, model_name, model_version
"""

# ── Build findings context for follow-up grounding ──
findings_context = "\nCURRENT FINDINGS FROM THE EXECUTIVE BRIEFING:\n"
for f in findings:
    findings_context += f"- [{f.severity.upper()}] {f.title}: {f.description}\n"
    findings_context += f"  Evidence: {'; '.join(f.evidence[:3])}\n"


def extract_sql_from_malformed(response) -> str:
    """Extract SQL from a MALFORMED_FUNCTION_CALL response if possible."""
    metadata = getattr(response, 'response_metadata', {})
    finish_msg = metadata.get('finish_message', '')
    if 'sql_query' in finish_msg and "query=" in finish_msg:
        import re
        match = re.search(r"query=['\"](.*?)['\"]\)", finish_msg, re.DOTALL)
        if not match:
            match = re.search(r"query=\'\'\'(.*?)\'\'\'", finish_msg, re.DOTALL)
        if match:
            return match.group(1).strip()
    return ""


def summarize_result(question: str, result_text: str) -> str:
    """Ask the LLM to produce an analyst-style summary of the SQL result."""
    summary_prompt = (
        f"You are an AI Business Analyst. A follow-up question was asked and answered via SQL.\n\n"
        f"Question: {question}\n\n"
        f"SQL Result:\n{result_text}\n\n"
        f"Write a 2-3 sentence analyst-style summary of the result. "
        f"Begin by connecting this to a relevant finding from the briefing if applicable. "
        f"Be specific — cite numbers from the result. Do not invent numbers not in the result.\n\n"
        f"CURRENT FINDINGS:\n{findings_context}"
    )
    try:
        response = llm.invoke([
            SystemMessage(content="You are a business analyst. Summarize data results concisely. Only reference numbers from the provided result."),
            HumanMessage(content=summary_prompt),
        ])
        summary = response.content if hasattr(response, "content") and response.content else ""
        if summary:
            return f"{summary}\n\n**Data:**\n{result_text}"
        return result_text
    except Exception:
        return result_text


def ask_followup(question: str) -> str:
    """Ask the agent a follow-up question grounded in the briefing context."""
    messages = [
        SystemMessage(content=(
            "You are an AI Business Analyst. An executive briefing was just generated. "
            "The user is asking a follow-up question about the findings. "
            "Use the sql_query tool to query the data and answer the question. "
            "You MUST use ONLY the exact column names and value types listed in the schema below. "
            "Do NOT guess or invent column names or values.\n"
            f"{TABLE_SCHEMAS}\n"
            f"{findings_context}"
        )),
        HumanMessage(content=f"Question: {question}"),
    ]

    response = llm_with_tools.invoke(messages)

    # ── Try structured tool calls first ──
    raw_result = ""
    output_parts = []
    if hasattr(response, "content") and response.content:
        output_parts.append(response.content)

    if hasattr(response, "tool_calls") and response.tool_calls:
        for tc in response.tool_calls:
            tool_fn = {t.name: t for t in agent_tools}.get(tc["name"])
            if tool_fn:
                try:
                    result = tool_fn.invoke(tc["args"])
                    raw_result = result
                    output_parts.append(f"\n[{tc['name']}]\n{result}")
                except Exception as e:
                    output_parts.append(f"\n[{tc['name']}] Error: {e}")

    # ── Fallback: extract SQL from malformed function call ──
    if not output_parts or (len(output_parts) == 1 and not output_parts[0].strip()):
        extracted_sql = extract_sql_from_malformed(response)
        if extracted_sql:
            try:
                result_df = con.execute(extracted_sql).fetchdf()
                if not result_df.empty:
                    raw_result = result_df.to_string(index=False, max_rows=20, float_format=lambda x: f"{x:,.2f}")
                    output_parts = [raw_result]
                else:
                    output_parts = ["Query returned no results."]
            except Exception as e:
                output_parts = [f"SQL Error: {e}\nAttempted query: {extracted_sql}"]
        else:
            metadata = getattr(response, 'response_metadata', {})
            finish_msg = metadata.get('finish_message', '')
            if finish_msg:
                output_parts = [f"Agent response: {finish_msg}"]
            else:
                output_parts = ["The agent could not generate a response for this question."]

    # ── Analyst-style summary: pass result back to LLM for narrative ──
    raw_output = "\n".join(output_parts)
    if raw_result and "Error" not in raw_result and "no results" not in raw_result.lower():
        return summarize_result(question, raw_result)
    return raw_output


# ── Example follow-up questions ──
followup_questions = [
    "Which distribution centers have the highest average delivery times?",
    "Show me the top 5 product categories by revenue.",
    "What percentage of high-risk churn customers came from each traffic source?",
]

for i, question in enumerate(followup_questions, 1):
    print("=" * 60)
    print(f"  Follow-up {i}: {question}")
    print("=" * 60)
    answer = ask_followup(question)
    print(answer)
    print()

  Follow-up 1: Which distribution centers have the highest average delivery times?
The distribution centers with the highest average delivery times are Los Angeles CA, with averages of 7.00 and 6.00 days, followed by Mobile AL, Charleston SC, and Chicago IL, all with an average of 6.00 days. This contrasts with the Port Authority of New York/New Jersey NY/NJ and Savannah GA, which show the lowest average delivery times at 1.00 day and -1.00 day respectively.

**Data:**
                   distribution_center_name  avg_delivery_days
                             Los Angeles CA               7.00
                             Los Angeles CA               6.00
                                  Mobile AL               6.00
                              Charleston SC               6.00
                                 Houston TX               5.00
                             New Orleans LA               5.00
                                 Chicago IL               5.00
                      

### Summary

This notebook demonstrates a complete AI Business Analyst that:

1. **Computes** — Deterministic analytics via DuckDB against Gold tables. No LLM involvement.
2. **Detects** — Threshold-based anomaly flagging with confidence scores and evidence trails.
3. **Investigates** — LangGraph agent conditionally drills into critical findings using tools.
4. **Synthesizes** — LLM generates a structured executive briefing from pre-computed signals.
5. **Follows Up** — Interactive Q&A grounded in briefing context, using SQL and visualization tools.

**Key design principle:** The LLM never computes metrics, Python and DuckDB handle all analytics. The LLM reasons over structured signals and generates narratives.
